## Download Audio and Playlsit metadata
Gets Audiofiles which are .opus format and metadata as json. Appends a load time to it. <br>
Note treats playlist metadata as a source of truth for what is on disk. 

In [1]:
import os
import yt_dlp
import pandas as pd
import json
from pathlib import Path
from pyprojroot.here import here
import subprocess

# EMF 2026 audio collection
PLAYLIST_URL = "https://media.ccc.de/c/emf2026/audio"

# Destination folder
ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
DOWNLOAD_FOLDER = ROOT / "Data"/ "Landing_Zone" / "Recordings"
METADATA_FILE_PATH = DOWNLOAD_FOLDER.parent / "recording_metadata.json"

print(DOWNLOAD_FOLDER.resolve())

C:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\Recordings


In [2]:
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)


ydl_opts = {
    "format": "bestaudio/best", # Audio Quality. Usually .opus
    "outtmpl": str(DOWNLOAD_FOLDER / "%(id)s - %(title)s.%(ext)s"), # Filename template
    "yes_playlist": True, # Download entire playlist
    "ignoreerrors": True, # Avoid stopping if an individual download fails
    "overwrites": False, # Do not overwrite existing media files
    "download_archive": str(DOWNLOAD_FOLDER / "downloaded_archive.txt"),  # Option specifies a location for yt-dlp to create a text file format: "source id\n", it uses this to avoid re-downloading files that have already been downloaded. e.g "ccc 17952"
    "progress": True, # Show progress 
}


print("EMF 2026 Audio Downloader")
print("-------------------------")
print(f"Destination: {DOWNLOAD_FOLDER}")
print("Starting download...\n")

# One extraction pass downloads audio AND return playlist metadata
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    playlist_info = ydl.extract_info(PLAYLIST_URL, download=True)
    

entries = (playlist_info or {}).get("entries") or [] # Unsual clever usecase for "or" as a denser alternatve for if/else. Checks statment in () first. Then...
                                                     # If playlist info has content it prefers that. Playlist info = none treated like False so defaults to {}
                                                     # Same Boolean like procedure occurs with {}.get(entries) returning none or values. if none then entries gets empty list. 

load_time = pd.Timestamp.now().isoformat()

valid_entries = [ # filtered list comprehension. 
    {**entry, "loaded_at": load_time}       # What gets added to the new list. Entry is unpacked and new key value pair added to each item at root depth. 
    for entry in entries                    # Loop through each item in `entries`
    if isinstance(entry, dict) and entry    # Filter: keep only non-empty dicts
]

# Write To file now with load_time included.
if valid_entries:
    output_payload = {
        **(playlist_info if isinstance(playlist_info, dict) else {}),
        "entries": valid_entries,
    }
    with open(METADATA_FILE_PATH, "w", encoding="utf-8") as f:
        json.dump(output_payload, f, indent=2, ensure_ascii=False, default=str)
    print(f"Saved metadata for {len(valid_entries)} recordings to: {METADATA_FILE_PATH}")
else:
    print("No valid metadata returned; skipping recording_metadata.json update.")


print("\nDownload process finished.")
print(f"Audio files saved in: {DOWNLOAD_FOLDER}")

EMF 2026 Audio Downloader
-------------------------
Destination: C:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\Recordings
Starting download...

[media.ccc.de:lists] Extracting URL: https://media.ccc.de/c/emf2026/audio
[media.ccc.de:lists] emf2026: Downloading JSON metadata
[download] Downloading playlist: Electromagnetic Field 2026
[media.ccc.de:lists] Playlist Electromagnetic Field 2026: Downloading 130 items of 130
[download] Downloading item 1 of 130
[media.ccc.de] Extracting URL: https://media.ccc.de/v/emf2026-240-1-men-in-tech-yesallmen
[media.ccc.de] emf2026-240-1-men-in-tech-yesallmen: Downloading webpage
[media.ccc.de] 17954: Downloading JSON metadata
[download] 17954: Men in tech #YesAllMen has already been recorded in the archive
[download] Downloading item 2 of 130
[media.ccc.de] Extracting URL: https://media.ccc.de/v/emf2026-290-1-endocrine-biohacking
[media.ccc.de] emf2026-290-1-endocrine-biohacking: Downloading webpage
[media.ccc.de] 17952: Downlo